In [1]:
"""
11_train_window_sensitivity.py

Robustness check: does the length of the training window matter?

07-10 train on 2 bookyears (2017+2018, config.TRAIN_YEARS) and test on
2019. This notebook refits the same 4 models using only 1 training
year (2018 - the year closest to the 2019 test year, and comparable to
a single-year setup like Cultrera & Bredart's), and compares the
result side by side with the 2-year setup.

To avoid duplicate work, the 2-year numbers are NOT recomputed here -
they are read back from logs/model_results.csv, where 07-10 already
logged them under their plain model name (e.g. "XGBoost Financial").
Only the 1-year scenario is fit fresh, logged under a name with a
"(train=2018)" suffix so it's clearly distinguishable.

Run 07-10 before this notebook.
"""


'\n11_train_window_sensitivity.py\n\nRobustness check: does the length of the training window matter?\n\n07-10 train on 2 bookyears (2017+2018, config.TRAIN_YEARS) and test on\n2019. This notebook refits the same 4 models using only 1 training\nyear (2018 - the year closest to the 2019 test year, and comparable to\na single-year setup like Cultrera & Bredart\'s), and compares the\nresult side by side with the 2-year setup.\n\nTo avoid duplicate work, the 2-year numbers are NOT recomputed here -\nthey are read back from logs/model_results.csv, where 07-10 already\nlogged them under their plain model name (e.g. "XGBoost Financial").\nOnly the 1-year scenario is fit fresh, logged under a name with a\n"(train=2018)" suffix so it\'s clearly distinguishable.\n\nRun 07-10 before this notebook.\n'

In [2]:
from utils.load_data_features import load_data_features

df = load_data_features()


In [3]:
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

from config import (
    FEATURES_BEHAVIORAL, FEATURES_FINANCIAL, RANDOM_STATE, TARGET,
    TRAIN_YEARS, WINSOR_COLUMNS,
)
from utils.model_results import load_model_results, save_model_results
from utils.print_section import print_section
from utils.time_based_split import time_based_split
from utils.winsorizer import Winsorizer

ONE_YEAR = [2018]
METRIC_COLUMNS = ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]

FEATURE_SETS = {
    "Financial": FEATURES_FINANCIAL,
    "Financial + Behavioral": FEATURES_FINANCIAL + FEATURES_BEHAVIORAL,
}

# (result label, which model, which feature set) - "result label" must
# match the plain model name 07-10 log under (MODEL_NAME in those
# notebooks), so their 2017-2018 results can be looked up below.
MODEL_SPECS = [
    ("Logistic Regression", "Logistic Regression", "Financial"),
    ("Random Forest", "Random Forest", "Financial"),
    ("XGBoost Financial", "XGBoost", "Financial"),
    ("XGBoost Financial + Behavioral", "XGBoost", "Financial + Behavioral"),
]


def make_pipeline(model_key, y_train):
    """Build the same pipeline as in 07-10, given which model to use.

    scale_pos_weight for XGBoost depends on y_train, so it has to be
    computed for this training window, not reused from 07-10.
    """
    if model_key == "Logistic Regression":
        model = LogisticRegression(
            max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE,
        )
    elif model_key == "Random Forest":
        model = RandomForestClassifier(
            n_estimators=500, random_state=RANDOM_STATE,
            class_weight="balanced_subsample", n_jobs=-1,
        )
    elif model_key == "XGBoost":
        scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
        model = XGBClassifier(
            n_estimators=500, max_depth=4, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            scale_pos_weight=scale_pos_weight,
            random_state=RANDOM_STATE, eval_metric="logloss",
        )
    else:
        raise ValueError(f"Unknown model_key: {model_key}")

    return Pipeline([
        ("winsorizer", Winsorizer(columns=WINSOR_COLUMNS)),
        ("imputer", SimpleImputer(strategy="median")),
        ("model", model),
    ])


# ------------------------------------------------------------------
# Fit the 1-year (2018) scenario for each model.
# ------------------------------------------------------------------
print_section("Fitting 1-year (2018) scenario")

one_year_rows = []

for result_label, model_key, feature_set_key in MODEL_SPECS:
    features = FEATURE_SETS[feature_set_key]

    X_train, X_test, y_train, y_test = time_based_split(
        df, features, TARGET, train_years=ONE_YEAR,
    )

    pipeline = make_pipeline(model_key, y_train)
    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_prob),
        "pr_auc": average_precision_score(y_test, y_prob),
    }

    save_model_results(f"{result_label} (train=2018)", metrics)
    one_year_rows.append({"model": result_label, "train_window": "2018", **metrics})

    print(
        f"{result_label:35s} "
        f"roc_auc={metrics['roc_auc']:.4f} "
        f"pr_auc={metrics['pr_auc']:.4f} "
        f"recall={metrics['recall']:.4f} "
        f"precision={metrics['precision']:.4f}"
    )

one_year_df = pd.DataFrame(one_year_rows)

# ------------------------------------------------------------------
# Reuse the 2017-2018 results already logged by 07-10, instead of
# refitting the exact same models a second time.
# ------------------------------------------------------------------
print_section("Reusing existing 2017-2018 results (from 07-10)")

canonical_names = [spec[0] for spec in MODEL_SPECS]
all_results = load_model_results()

two_year_df = all_results[all_results["model"].isin(canonical_names)][["model"] + METRIC_COLUMNS].copy()
two_year_df["train_window"] = "2017-2018"

missing = set(canonical_names) - set(two_year_df["model"])
if missing:
    raise RuntimeError(
        f"No 2017-2018 results found for {missing}. Run notebooks 07-10 first."
    )

print(f"Reused {len(two_year_df)} results, no refitting needed.")



Fitting 1-year (2018) scenario


Logistic Regression                 roc_auc=0.8026 pr_auc=0.0098 recall=0.6478 precision=0.0068


Random Forest                       roc_auc=0.7576 pr_auc=0.0107 recall=0.0032 precision=0.0155


XGBoost Financial                   roc_auc=0.8584 pr_auc=0.0201 recall=0.7011 precision=0.0085


XGBoost Financial + Behavioral      roc_auc=0.8586 pr_auc=0.0233 recall=0.7076 precision=0.0088

Reusing existing 2017-2018 results (from 07-10)
Reused 4 results, no refitting needed.


In [4]:
# ------------------------------------------------------------------
# Side-by-side comparison: 1 training year vs. 2 training years,
# same model, same test year (2019).
# ------------------------------------------------------------------
print_section("1 year vs. 2 years - side by side")

combined = pd.concat([one_year_df, two_year_df], ignore_index=True)

comparison = combined.pivot(
    index="model", columns="train_window", values=["roc_auc", "pr_auc", "recall", "precision"],
)

print(comparison.round(4))

# ------------------------------------------------------------------
# Training set size per window (row counts only, no fitting needed).
# ------------------------------------------------------------------
print_section("Training set size per window")

size_rows = []
for window_label, train_years in {"2018": ONE_YEAR, "2017-2018": TRAIN_YEARS}.items():
    _, _, y_train_window, _ = time_based_split(df, FEATURES_FINANCIAL, TARGET, train_years=train_years)
    size_rows.append({
        "train_window": window_label,
        "train_rows": len(y_train_window),
        "train_failures": int(y_train_window.sum()),
    })

print(pd.DataFrame(size_rows))



1 year vs. 2 years - side by side
                                 roc_auc            pr_auc            recall  \
train_window                   2017-2018    2018 2017-2018    2018 2017-2018   
model                                                                          
Logistic Regression               0.8024  0.8026    0.0097  0.0098    0.6527   
Random Forest                     0.7594  0.7576    0.0109  0.0107    0.0048   
XGBoost Financial                 0.8631  0.8584    0.0233  0.0201    0.7383   
XGBoost Financial + Behavioral    0.8637  0.8586    0.0249  0.0233    0.7399   

                                       precision          
train_window                      2018 2017-2018    2018  
model                                                     
Logistic Regression             0.6478    0.0065  0.0068  
Random Forest                   0.0032    0.0160  0.0155  
XGBoost Financial               0.7011    0.0078  0.0085  
XGBoost Financial + Behavioral  0.7076    0.0081  

  train_window  train_rows  train_failures
0         2018      336357            1079
1    2017-2018      656709            2203
